# Surrogate MGGP + PSO — Himmelblau 2D
**Surrogate-assisted optimization**: MGGP fits a surrogate on 300 LHS samples,
then PSO minimizes on the surrogate surface.

New library features showcased (not used in other notebooks):
- `combiner="lasso"` — Lasso regression combiner
- `RouletteSelection` — fitness-proportionate roulette selection
- `regression_degree=2` — degree-2 polynomial gene features
- `n_genes_max` — upper bound on gene count during evolution
- `elite_ratio` — fraction of elite individuals preserved each generation

Figures produced:
- `fig02_surrogate_obs_pred.png` — Surrogate quality (Observed × Predicted)
- `fig02_surrogate_surface.png` — 3D: true Himmelblau vs MGGP surrogate surface
- `fig02_pso_himmelblau_contour.png` — PSO spatial distribution: true vs surrogate
- `fig02_pso_convergence_surrogate.png` — PSO convergence on the surrogate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
from sklearn.metrics import r2_score
import os, math

FIG_DIR = Path("figures")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_COLOR = '#1f77b4'
VAL_COLOR   = '#ff7f0e'
TEST_COLOR  = '#2ca02c'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.linestyle': '-', 'grid.alpha': 0.4,
    'grid.color': '#cccccc', 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 13, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

def obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title,
                xlabel="Observed", ylabel="Predicted"):
    h1 = ax.scatter(y_tr, yp_tr, c=TRAIN_COLOR, marker='o', s=40, alpha=0.75,
                    label=f'Train  (R²={r2_score(y_tr, yp_tr):.6f})')
    h2 = ax.scatter(y_v,  yp_v,  c=VAL_COLOR,   marker='s', s=40, alpha=0.75,
                    label=f'Val    (R²={r2_score(y_v,  yp_v):.6f})')
    h3 = ax.scatter(y_te, yp_te, c=TEST_COLOR,  marker='^', s=40, alpha=0.75,
                    label=f'Test   (R²={r2_score(y_te, yp_te):.6f})')
    all_y = np.concatenate([y_tr, y_v, y_te])
    lo, hi = all_y.min(), all_y.max()
    ax.plot([lo, hi], [lo, hi], color='black', lw=1.5)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    return h1, h2, h3

print("Style loaded.")

def _snap(ax, log_y=False):
    ax.figure.canvas.draw()
    ax.tick_params(top=True, right=True, which='both', direction='in')
    lo, hi = ax.get_xlim()
    xt = sorted(t for t in ax.get_xticks() if lo - 1e-9 <= t <= hi + 1e-9)
    if len(xt) >= 2:
        ax.set_xlim(xt[0], xt[-1])
    if log_y:
        lo, hi = ax.get_ylim()
        if lo > 0 and hi > 0:
            lo_dec = 10 ** math.floor(math.log10(lo))
            hi_log = math.log10(hi)
            frac   = hi_log - math.floor(hi_log)
            hi_dec = 10 ** (math.floor(hi_log) if frac < 0.02 else math.ceil(hi_log))
            if lo_dec < hi_dec:
                ax.set_ylim(lo_dec, hi_dec)
    else:
        lo, hi = ax.get_ylim()
        yt = sorted(t for t in ax.get_yticks() if lo - 1e-9 <= t <= hi + 1e-9)
        if len(yt) >= 2:
            ax.set_ylim(yt[0], yt[-1])

def _fix_cbar(cb, cp):
    _lo, _hi = cp.get_clim()
    _t = [x for x in cb.get_ticks() if _lo < x < _hi]
    cb.set_ticks([_lo] + _t + [_hi])


## Data Generation (70/15/15 split from 300 LHS samples)

In [3]:
from symgene.benchmarks import himmelblau_2d

bench = himmelblau_2d()
rng   = np.random.default_rng(0)

def lhs(rng, lo, hi, n):
    d = len(lo)
    X = np.empty((n, d))
    for j in range(d):
        perm = rng.permutation(n)
        X[:, j] = lo[j] + (perm + rng.uniform(0, 1, n)) / n * (hi[j] - lo[j])
    return X

lo, hi = np.array([-5., -5.]), np.array([5., 5.])
X_all  = lhs(rng, lo, hi, 300)
y_all  = np.array([bench.fn(X_all[i]) for i in range(300)])

n_train, n_val = 210, 45           # 70 / 15 / 15
X_train = X_all[:n_train]
y_train = y_all[:n_train]
X_val   = X_all[n_train:n_train + n_val]
y_val   = y_all[n_train:n_train + n_val]
X_test  = X_all[n_train + n_val:]
y_test  = y_all[n_train + n_val:]

print(f"Train:{X_train.shape[0]}  Val:{X_val.shape[0]}  Test:{X_test.shape[0]}")
print(f"y range: [{y_all.min():.2f}, {y_all.max():.2f}]")


Train:210  Val:45  Test:45
y range: [1.07, 612.84]


## Fit MGGP Surrogate

Key hyperparameters — all **new** relative to the Ackley notebook:

| Parameter | Value | Description |
|---|---|---|
| `combiner` | `"lasso"` | Lasso regression (L1 sparsity) instead of Ridge |
| `selection` | `RouletteSelection()` | Fitness-proportionate selection |
| `regression_degree` | `2` | Polynomial features of gene outputs |
| `n_genes_max` | `8` | Maximum gene count during evolution |
| `elite_ratio` | `0.04` | 4% elite preserved each generation |

In [4]:
from symgene import SymGeneRegressor
from symgene.primitives import STANDARD
from symgene.selection.roulette import RouletteSelection
from symgene.metrics import mae, r2, mape

regressor = SymGeneRegressor(
    n_genes=3,
    pop_size=80,
    n_gen=100,
    primitives=STANDARD,
    squash={"lim": 8, "alpha": 0.08, "scale": 2.0},
    feature_names=["x1", "x2"],
    seed=42,
    verbose=0,
    # ── New library features showcased ───────────────────────────────────────
    combiner="lasso",              # <<< Lasso: L1 regularisation, promotes sparsity
    regression_degree=2,           # <<< degree-2 polynomial of gene outputs
    selection=RouletteSelection(), # <<< fitness-proportionate roulette selection
    n_genes_max=8,                 # <<< upper bound on gene count
    elite_ratio=0.04,              # <<< 4% elite preserved each generation
    # ── Remaining operators ───────────────────────────────────────────────────
    mutpb=0.28, mutpb_low=0.18,
    mutation_weights=[0.4, 1.5, 0.8],
    tree_max=30,
    height_max=6,
)
regressor.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("MGGP surrogate fitted.")
print(f"Genes     : {regressor.n_genes_}")
print(f"Expression: {regressor.best_expression_}")


MGGP surrogate fitted.
Genes     : 8
Expression: min3(log(sqrt(x1)), atan(div(x2, x1)), mean3(x2, add(x1, x2), sqrt(x2))) | sin(sub(sqrt(add(relu(cos(x2)), x2)), x2)) | sub(min3(add(x2, cos(x1)), x2, x2), min2(sigmoid(x1), min3(x2, x1, sub(abs(x1), div(x1, x1))))) | sin(mean2(mean3(max2(log(min3(x1, x1, x2)), x1), relu(sin(x2)), max2(x2, x1)), x2)) | mul(x2, log(min3(add(x2, abs(cos(x2))), x2, x2))) | cos(mean2(log(sqrt(mean2(relu(log(x2)), abs(x1)))), x1)) | exp(relu(log(add(x2, abs(sin(x1)))))) | mul(div(x2, x2), cos(x1))


## Run PSO on Surrogate

In [5]:
from symgene.optimization import PSOOptimizer

optimizer = PSOOptimizer(n_particles=80, n_iter=300, verbose=0)

def surrogate_fn(x):
    return float(regressor.predict(x.reshape(1, -1))[0])

pso_result = optimizer.optimize(surrogate_fn, bounds=bench.bounds, seed=0)
f_true = float(bench.fn(pso_result.x_best))

known_optima = [
    ( 3.000000,  2.000000),
    (-2.805118,  3.131312),
    (-3.779310, -3.283186),
    ( 3.584428, -1.848126),
]

print(f"PSO found   : x={np.round(pso_result.x_best, 4)}  f_surrogate={pso_result.f_best:.4f}")
print(f"True f(x*)  : {f_true:.6f}")
print(f"True optima : f=0 at 4 points (Himmelblau)")
print(f"Gap to nearest optimum: {min(np.hypot(pso_result.x_best[0]-ox, pso_result.x_best[1]-oy) for ox,oy in known_optima):.6f}")


PSO found   : x=[2.8537 1.8537]  f_surrogate=1.3220
True f(x*)  : 1.510270
True optima : f=0 at 4 points (Himmelblau)
Gap to nearest optimum: 0.206944


## Figure 1 — Surrogate Quality: Observed × Predicted

In [ ]:
yp_train = regressor.predict(X_train)
yp_val = regressor.predict(X_val)
yp_test = regressor.predict(X_test)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

obs_pred_ax(
    ax,
    y_train,
    yp_train,
    y_val,
    yp_val,
    y_test,
    yp_test,
    title="Surrogate Quality — Observed × Predicted",
    xlabel="Observed Himmelblau f(x)",
    ylabel="Surrogate Prediction",
)

# --- 1. Remove borders/lines from plotted points ---
# For markers generated via ax.scatter
for collection in ax.collections:
    collection.set_linewidth(0)
    collection.set_edgecolor("none")

# For markers generated via ax.plot
for line in ax.lines:
    if line.get_marker() != "None" and line.get_marker() != "":
        line.set_linewidth(0)
        line.set_markeredgewidth(0)

# --- 2. Apply _snap ---
_snap(ax)

# --- 3. Add grid ---
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

# --- 4. Black borders and axes ---
ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

# --- 5. Clean legend (no lines through icons) ---
leg = ax.legend(
    loc="upper left",
    frameon=True,
    edgecolor="black",
    fancybox=False,
    fontsize=9,
    handlelength=1.2,
    handletextpad=0.5,
)

for handle in leg.legend_handles:
    if hasattr(handle, "set_linewidth"):
        handle.set_linewidth(0)

plt.savefig(os.path.join(FIG_DIR, "fig02_surrogate_obs_pred.png"), dpi=300)
plt.show()
print("Saved: fig02_surrogate_obs_pred.png")

## Figure 2 — 3D Surface: True Himmelblau vs MGGP Surrogate
Left: true Himmelblau 2D analytical function (cmap `viridis`).
Right: surrogate surface learned by MGGP (cmap `plasma`).
The visual similarity demonstrates the surrogate's ability to capture the multimodal landscape.

In [ ]:
def himmelblau_vec(x1, x2):
    return (x1**2 + x2 - 11)**2 + (x1 + x2**2 - 7)**2

# Grid for both surfaces
x1g = np.linspace(-5, 5, 70)
x2g = np.linspace(-5, 5, 70)
X1g, X2g = np.meshgrid(x1g, x2g)
X_grid = np.column_stack([X1g.ravel(), X2g.ravel()])

Z_true = himmelblau_vec(X1g, X2g)
Z_surr = regressor.predict(X_grid).reshape(X1g.shape)

ELEV, AZIM = 28, -55

fig, axes = plt.subplots(1, 2, figsize=(15, 7),
                         subplot_kw={"projection": "3d"},
                         gridspec_kw={'wspace': 0})

plt.suptitle("Himmelblau 2D — True Surface vs MGGP Surrogate", fontsize=14, y=0.98)

# ── True surface ───────────────────────────────────────────────────────────
axes[0].plot_surface(X1g, X2g, Z_true, cmap="viridis",
                     alpha=0.92, linewidth=0, antialiased=True)
axes[0].set_title("True Function", fontsize=12, pad=10)
axes[0].set_xlabel("$x_1$", labelpad=1)
axes[0].set_ylabel("$x_2$", labelpad=1)
axes[0].set_zlabel("$f(x_1,x_2)$", fontsize=9, labelpad=0)

# Fine-tune Z axis (True)
axes[0].set_zlim(0, 900)
axes[0].set_zticks(np.linspace(0, 900, 10))  # Ticks from 0 to 900 in steps of 100

axes[0].view_init(elev=ELEV, azim=AZIM)

# ── MGGP surrogate surface ─────────────────────────────────────────────────
axes[1].plot_surface(X1g, X2g, Z_surr, cmap="plasma",
                     alpha=0.92, linewidth=0, antialiased=True)
axes[1].set_title(f"MGGP Surrogate  (R²(test)={r2(y_test, yp_test):.4f})", fontsize=12, pad=10)
axes[1].set_xlabel("$x_1$", labelpad=1)
axes[1].set_ylabel("$x_2$", labelpad=1)
axes[1].set_zlabel("$f(x_1,x_2)$", fontsize=9, labelpad=0)

# Fine-tune Z axis (Surrogate)
# Option A: Identical scales for both panels (recommended for surrogate)
axes[1].set_zlim(0, 900)
axes[1].set_zticks(np.linspace(0, 900, 10))

# Option B: Fix surrogate axis at 700:
# axes[1].set_zlim(0, 700)
# axes[1].set_zticks(np.linspace(0, 700, 8))

axes[1].view_init(elev=ELEV, azim=AZIM)

# Layout and save
plt.subplots_adjust(left=0.03, right=0.90, top=0.90, bottom=0.05)

output_file = os.path.join(FIG_DIR, "fig02_surrogate_surface.png")
plt.savefig(output_file, dpi=300, pad_inches=0.1)
plt.show()
print(f"Saved: {output_file}")

## Figure 3 — PSO Spatial Distribution: True vs Surrogate Landscape
Left: true Himmelblau 2D landscape (log scale, cmap `viridis`).
Right: MGGP surrogate landscape (cmap `plasma`).
Black stars = four true global optima (f=0).  Red diamond = best point found by PSO on the surrogate.

In [ ]:
def himmelblau_vec(x1, x2):
    return (x1**2 + x2 - 11) ** 2 + (x1 + x2**2 - 7) ** 2


x1g = np.linspace(-5, 5, 300)
x2g = np.linspace(-5, 5, 300)
X1g, X2g = np.meshgrid(x1g, x2g)
X_mesh = np.column_stack([X1g.ravel(), X2g.ravel()])

Z_true_c = himmelblau_vec(X1g, X2g)
Z_surr_c = regressor.predict(X_mesh).reshape(X1g.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plt.subplots_adjust(wspace=0.25)

panels = [
    (Z_true_c, "Himmelblau 2D — True Function", "viridis"),
    (Z_surr_c, "MGGP Surrogate — Estimated Landscape", "plasma"),
]

ox, oy = zip(*known_optima)

# --- Fixed and evenly spaced colorscale configuration ---
V_MIN = 0.0
V_MAX = 6.8
# Define 35 uniform fill levels
levels = np.linspace(V_MIN, V_MAX, 35)
# Define 5 evenly spaced colorbar ticks: [0.0, 1.7, 3.4, 5.1, 6.8]
cb_ticks = np.linspace(V_MIN, V_MAX, 5)

for ax, (Z, title, cmap) in zip(axes, panels):
    Z_log = np.log1p(np.clip(Z, 0, None))

    # Lock vmin and vmax so both panels share the same strict range
    cp = ax.contourf(
        X1g,
        X2g,
        Z_log,
        levels=levels,
        vmin=V_MIN,
        vmax=V_MAX,
        cmap=cmap,
        alpha=0.88,
    )

    # Colorbar with strict scale (no loose ticks at the top)
    _cb = plt.colorbar(
        cp,
        ax=ax,
        label="log(1 + f)",
        shrink=0.85,
        ticks=cb_ticks,  # Force only the linspace-calculated points
    )
    _fix_cbar(_cb, cp)

    # Four true global optima
    ax.scatter(
        ox,
        oy,
        color="black",
        marker="*",
        s=70,
        zorder=6,
        label="True optima  f=0  (4 pts)",
    )

    # PSO result
    ax.scatter(
        pso_result.x_best[0],
        pso_result.x_best[1],
        color="red",
        marker="D",
        s=15,
        zorder=7,
        edgecolors="black",
        lw=0.8,
        label=f"PSO found  f_true={f_true:.2e}",
    )

    ax.set_xlabel("$x_1$", color="black")
    ax.set_ylabel("$x_2$", color="black")
    ax.set_title(title, color="black")
    ax.legend(
        loc="upper right",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
    )

    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)

    if "_snap" in globals():
        _snap(ax)

    # Grid and black axes
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5, color="gray")

    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

plt.savefig(
    os.path.join(FIG_DIR, "fig02_pso_himmelblau_contour.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print("Saved: fig02_pso_himmelblau_contour.png")

## Figure 4 — PSO Convergence on the Surrogate Surface

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.semilogy(
    pso_result.history,
    color=TRAIN_COLOR,
    lw=2,
    label="PSO best (on surrogate)",
)
ax.set_xlabel("Iteration", color="black")
ax.set_ylabel("Best fitness on surrogate (log scale)", color="black")
ax.set_title(
    "PSO Convergence on MGGP Surrogate — Himmelblau 2D", color="black"
)
ax.legend(loc="upper right", frameon=True, edgecolor="black", fancybox=False)

# 1. Apply _snap first
_snap(ax, log_y=True)

# 2. Add grid (major and minor lines for log scale)
ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

# 3. Ensure black axes and borders
ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

plt.savefig(
    os.path.join(FIG_DIR, "fig02_pso_convergence_surrogate.png"), dpi=300
)
plt.show()
print("Saved: fig02_pso_convergence_surrogate.png")

## Numerical Results

In [10]:
print(f"Surrogate R\u00b2   (test) : {r2(y_test, yp_test):.4f}")
print(f"Surrogate MAE  (test) : {mae(y_test, yp_test):.4f}")
print(f"Surrogate MAPE (test) : {mape(y_test, yp_test):.2f}%")
print()
print(f"PSO found x       : {np.round(pso_result.x_best, 6)}")
print(f"Surrogate f(x*)   : {pso_result.f_best:.6f}")
print(f"True f(x*)        : {f_true:.6f}")
print(f"Known optima      : f=0 at 4 locations (Himmelblau)")
dist_to_nearest = min(np.hypot(pso_result.x_best[0]-ox, pso_result.x_best[1]-oy)
                      for ox, oy in known_optima)
print(f"Dist. to nearest  : {dist_to_nearest:.6f}")


Surrogate R²   (test) : 0.9835
Surrogate MAE  (test) : 10.3825
Surrogate MAPE (test) : 27.40%

PSO found x       : [2.853669 1.853669]
Surrogate f(x*)   : 1.322028
True f(x*)        : 1.510270
Known optima      : f=0 at 4 locations (Himmelblau)
Dist. to nearest  : 0.206944
